In [1]:
!pip install nvidia-tensorrt

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 17.6 MB/s eta 0:00:00a 0:00:01
  Created wheel for tensorrt: filename=tensorrt-10.15.1.29-py2.py3-none-any.whl size=16660 sha256=22b68985084264eb680605e260740af311369e6d53d2a9f4723398cfedbc7baa
  Stored in directory: /root/.cache/pip/wheels/b9/22/8d/fb7426b0beeea6ae8767cec6c61da3fa3e65b03d04f44ecf88
  Created wheel for tensorrt_cu13: filename=tensorrt_cu13-10.15.1.29-py2.py3-none-any.whl size=23170 sha256=33ca0a154b7eea34c8b48a16bb1c1c8d823be215867d48a8351c87726956d184
  Stored in directory: /root/.cache/pip/wheels/81/d1/19/167647f2f815acfae87b56fb4c436b6b3b8046c4f503828287
  Created wheel for tensorrt_cu13_libs: filename=tensorrt_cu13_libs-10.15.1.29-py2.py3-none-manylinux_2_28_x86_64.whl size=3713901680 sha256=0619b6

In [2]:
!pip install onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 10.8 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 11.2 MB/s eta 0:00:00


In [3]:
import torch
import torch.nn as nn
import tensorrt as trt
import os

In [4]:
class SCOnv(nn.Module):
    def __init__(self):
        super().__init__()
        self.c = nn.Conv2d(3, 16, kernel_size=3, padding=1)
    def forward(self, x):
        return self.c(x)

In [5]:
mod = SCOnv().cuda().eval()
dum = torch.randn(1, 3, 224, 224).cuda()

In [6]:
onnx_file = 'simple_conv.onnx'
if os.path.exists(onnx_file): os.remove(onnx_file)

torch.onnx.export(
    mod, dum, onnx_file,
    input_names=['input'],
    output_names=['output'],
    opset_version=13,
    do_constant_folding=True
)

W0224 14:11:42.328000 55 torch/onnx/_internal/exporter/_compat.py:114] Setting ONNX exporter to use operator set version 18 because the requested opset_version 13 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `SCOnv([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SCOnv([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 13).


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 13},
            producer_name='pytorch',
            producer_version='2.9.0+cu126',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"input"<FLOAT,[1,3,224,224]>
            ),
            outputs=(
                %"output"<FLOAT,[1,16,224,224]>
            ),
            initializers=(
                %"c.weight"<FLOAT,[16,3,3,3]>{TorchTensor(...)},
                %"c.bias"<FLOAT,[16]>{TorchTensor(...)}
            ),
        ) {
            0 |  # node_conv2d
                 %"output"<FLOAT,[1,16,224,224]> ⬅️ ::Conv(%"input", %"c.weight"{...}, %"c.bias"{...}) {group=1, pads=(1, 1, 1, 1), auto_pad='NOTSET', strides=(1, 1), dilations=(1, 1)}
            return %"output"<FLOAT,[1,16,224,224]>
        }


    ,
    exported_program=
        ExportedProgram:
            class GraphModule(torch.n

In [7]:
TRT_LOGGER = trt.Logger(trt.Logger.INFO)
builder = trt.Builder(TRT_LOGGER)
network = builder.create_network(1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))
parser = trt.OnnxParser(network, TRT_LOGGER)

[02/24/2026-14:11:47] [TRT] [I] [MemUsageChange] Init CUDA: CPU +0, GPU +0, now: CPU 220, GPU 106 (MiB)


In [8]:
with open(onnx_file, 'rb') as f:
    parser.parse(f.read())

[02/24/2026-14:11:47] [TRT] [I] ----------------------------------------------------------------
[02/24/2026-14:11:47] [TRT] [I] ONNX IR version:  0.0.10
[02/24/2026-14:11:47] [TRT] [I] Opset version:    13
[02/24/2026-14:11:47] [TRT] [I] Producer name:    pytorch
[02/24/2026-14:11:47] [TRT] [I] Producer version: 2.9.0+cu126
[02/24/2026-14:11:47] [TRT] [I] Domain:           
[02/24/2026-14:11:47] [TRT] [I] Model version:    0
[02/24/2026-14:11:47] [TRT] [I] Doc string:       
[02/24/2026-14:11:47] [TRT] [I] ----------------------------------------------------------------


In [9]:
config = builder.create_builder_config()
config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 28)

In [10]:
tactic_sources = (1 << int(trt.TacticSource.CUBLAS)) | (1 << int(trt.TacticSource.CUBLAS_LT))
config.set_tactic_sources(tactic_sources)

True

In [11]:
serialized_engine = builder.build_serialized_network(network, config)
runtime = trt.Runtime(TRT_LOGGER)
engine = runtime.deserialize_cuda_engine(serialized_engine)
context = engine.create_execution_context()

[02/24/2026-14:11:58] [TRT] [I] BuilderFlag::kTF32 is set but hardware does not support TF32. Disabling TF32.
[02/24/2026-14:11:59] [TRT] [I] [MemUsageChange] Init builder kernel library: CPU +89, GPU +2, now: CPU 502, GPU 108 (MiB)
[02/24/2026-14:11:59] [TRT] [I] BuilderFlag::kTF32 is set but hardware does not support TF32. Disabling TF32.
[02/24/2026-14:11:59] [TRT] [I] Local timing cache in use. Profiling results in this builder pass will not be stored.
[02/24/2026-14:11:59] [TRT] [I] Detected 1 inputs and 1 output network tensors.
[02/24/2026-14:11:59] [TRT] [I] Total Host Persistent Memory: 5664 bytes
[02/24/2026-14:11:59] [TRT] [I] Total Device Persistent Memory: 0 bytes
[02/24/2026-14:11:59] [TRT] [I] Max Scratch Memory: 0 bytes
[02/24/2026-14:11:59] [TRT] [I] Total Activation Memory: 0 bytes
[02/24/2026-14:11:59] [TRT] [I] Total Weights Memory: 2624 bytes
[02/24/2026-14:11:59] [TRT] [I] Engine generation completed in 0.235879 seconds.
[02/24/2026-14:11:59] [TRT] [I] [MemUsageSt

In [12]:
input_tensor = torch.randn(1, 3, 224, 224, device='cuda', dtype=torch.float32).contiguous()
output_tensor = torch.empty((1, 16, 224, 224), device='cuda', dtype=torch.float32).contiguous()

input_name = engine.get_tensor_name(0)
output_name = engine.get_tensor_name(1)

context.set_input_shape(input_name, input_tensor.shape)
context.set_tensor_address(input_name, input_tensor.data_ptr())
context.set_tensor_address(output_name, output_tensor.data_ptr())

# Use the current PyTorch stream handle
stream = torch.cuda.current_stream()
success = context.execute_async_v3(stream_handle=stream.cuda_stream)

if success:
    torch.cuda.synchronize()
    print("\nSUCCESS!")
    print(f"Output mean: {output_tensor.mean().item():.6f}")
else:
    print("\nExecution failed even without Cask.")

[02/24/2026-14:12:29] [TRT] [W] Using default stream in enqueueV3() may lead to performance issues due to additional calls to cudaStreamSynchronize() by TensorRT to ensure correct synchronization. Please use non-default stream instead.

SUCCESS!
Output mean: -0.016474
